# DNN price forecasting on Colab

Runs the epftoolbox DNN (Lago, Marcjasz, De Schutter & Weron 2021) on a
bidding zone, using Colab's GPU.

**Set the runtime first:** Runtime → Change runtime type → GPU.

### Read this before you rely on the GPU

This workload is not compute-bound. Measured on one real recalibration:
218 epochs at 0.36 s each, of which **60% is the per-epoch validation
pass**, on a 224→216→24 network with roughly 1,100 training samples. The
matrix multiplies are trivial; the cost is per-call overhead in the custom
training loop, which a faster device does not remove. Expect a useful
speedup, not a tenfold one, and measure it below rather than assuming it.

## 1. Setup

The runtime is wiped on disconnect, so everything worth keeping goes to Drive.

In [39]:
!git clone https://github.com/Freddy5445/EPF_Masters.git
%cd EPF_Masters

Cloning into 'EPF_Masters'...
remote: Enumerating objects: 428, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 428 (delta 23), reused 20 (delta 18), pack-reused 396 (from 1)
Receiving objects: 100% (428/428), 13.74 MiB | 16.19 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/EPF_Masters/EPF_Masters/EPF_Masters/EPF_Masters


In [40]:
import colab_setup

# Installs what Colab lacks, installs the vendored epftoolbox from the local
# tree, mounts Drive, and reports what TensorFlow can actually use.
has_gpu = colab_setup.bootstrap()

-- dependencies
-- epftoolbox
-- epftoolbox: /content/EPF_Masters/epftoolbox/epftoolbox/__init__.py
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

TensorFlow 2.20.0
  built for CUDA 12.5.1, compute capabilities ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90']
  GPU: NVIDIA L4 (sm_89)


## 2. Point at your data

`datasets/*.csv` and the parquet are gitignored, so the clone has no data in it.

Put **`nordic_baltic_clean_hourly.parquet`** — what `data_cleaning.ipynb` writes —
on Drive at the `DRIVE` path below. One file covers every zone; the next section
projects whichever zone you want into the epftoolbox CSV layout inside the
runtime.

### Missing values

The projected CSV keeps its gaps: ENTSO-E does not publish everything, and a
naive local grid has no value at all for the spring-forward hour. The DNN cannot
be fitted on NaN, so `run_dnn_dk1.py` fills them with `lear_dk1.impute` — the
same module, the same 3-hour forward-fill default, as the LEAR runs. Every filled
value comes from **earlier observations only**; nothing is interpolated across a
gap and nothing looks forward.

Leading hours with no history behind them cannot be filled causally at all, so
they are dropped rather than invented, and the count is recorded.

What is imputed is training input. The **observed prices a forecast is scored
against are never imputed** — scoring against a filled value would measure
agreement with the filling rule instead of with the market. Hours with no
observed price are excluded from the metrics and counted in `evaluation.json`.

Pass `--no-impute` to see the gaps reported and stop instead.

In [41]:
import os

DRIVE = '/content/drive/MyDrive/EPF_Masters'
DATASETS = os.path.join(DRIVE, 'datasets')
OUT      = os.path.join(DRIVE, 'experiments')
DATASET  = 'DK1_clean_load-windsolar'

os.makedirs(DATASETS, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('datasets:', os.listdir(DATASETS) or '(empty - upload the CSV here)')

datasets: ['DK1_clean_load-windsolar.csv']


### Getting the data there

`files.upload()` sends the file through the notebook's websocket, base64-encoded,
which is slow almost regardless of size. Avoid it for anything real.

**Put the file in Drive instead** — drive.google.com in a browser, or the Drive
desktop client — and it is simply there when Drive is mounted. That path uses
proper resumable uploads and survives a dropped connection.

**Upload the parquet, not the CSVs.** `nordic_baltic_clean_hourly.parquet` holds
every zone and every series in one compressed columnar file; a CSV holds one zone
as uncompressed text. One upload then covers the whole sweep, and the next cell
regenerates any zone's CSV inside the runtime in seconds.

It also removes a way to get the comparison wrong: the CSV the DNN reads is
produced by the same projection the LEAR runs use, so the two models cannot
silently end up on differently built inputs.

In [42]:
# Build the zone CSV from the parquet, in the runtime.
# Same projection run_lear_from_clean uses for the LEAR runs, so both models
# read identical inputs.
import subprocess, sys

PANEL = os.path.join(DRIVE, 'nordic_baltic_clean_hourly.parquet')
ZONE, EXOG = 'DK1', 'load-windsolar'

if os.path.exists(PANEL):
    subprocess.run([sys.executable, 'run_lear_from_clean.py',
                    '--panel', PANEL, '--zone', ZONE, '--exog', EXOG,
                    '--datasets-dir', DATASETS, '--csv-only'], check=True)
    DATASET = f'{ZONE}_clean_{EXOG}'
    print('\ndataset:', DATASET)
else:
    print(f'No parquet at {PANEL}.')
    print('Put nordic_baltic_clean_hourly.parquet there, or set DATASET to a CSV '
          'already in', DATASETS)


dataset: DK1_clean_load-windsolar


#### Last resort: upload through the notebook

Only for a single small file, and only if Drive is not an option. Expect it to be
slow.

In [ ]:
from google.colab import files
import shutil

for name in files.upload():
    shutil.move(name, os.path.join(DATASETS, name))
    print('->', os.path.join(DATASETS, name))

## 3. Smoke test first

Five hyperparameter evaluations and three forecast days. This proves the
pipeline runs on this runtime. It says nothing about accuracy — five
evaluations is not a search.

In [46]:
# Built as a list rather than a ! shell line, because the paths are
# Python variables and a shell cell cannot see them.
import subprocess, sys

cmd = [sys.executable, 'run_dnn_dk1.py', '--smoke',
       '--dataset', DATASET, '--datasets-dir', DATASETS, '--out-dir', OUT]
print(' '.join(cmd))
subprocess.run(cmd, check=False)

/usr/bin/python3 run_dnn_dk1.py --smoke --dataset DK1_clean_load-windsolar --datasets-dir /content/drive/MyDrive/EPF_Masters/datasets --out-dir /content/drive/MyDrive/EPF_Masters/experiments


CompletedProcess(args=['/usr/bin/python3', 'run_dnn_dk1.py', '--smoke', '--dataset', 'DK1_clean_load-windsolar', '--datasets-dir', '/content/drive/MyDrive/EPF_Masters/datasets', '--out-dir', '/content/drive/MyDrive/EPF_Masters/experiments'], returncode=0)

## 4. Measure before committing hours

Time one recalibration on this runtime. Multiply by the number of forecast
days and seeds to get the real cost of the full run before starting it.

The reference figure from a CPU sandbox was **77.6 s** per recalibration.

In [48]:
import time, pandas as pd
from dnn_dk1 import DNN

hyper_dir = os.path.join(OUT, 'hyperparameters')
data = pd.read_csv(os.path.join(DATASETS, DATASET + '.csv'),
                   index_col=0, parse_dates=True)
data.columns = ['Price'] + [f'Exogenous {i}' for i in range(1, len(data.columns))]

model = DNN(path_hyperparameter_folder=hyper_dir, dataset=DATASET,
            calibration_window=4, seed=1)

day = data.index[-1].normalize()
started = time.time()
_ = model.recalibrate_and_forecast_next_day(data, day)
per_day = time.time() - started

print(f'one recalibration: {per_day:.1f}s')
for days, seeds in ((728, 4), (728, 1), (104, 4)):
    hours = per_day * days * seeds / 3600
    print(f'  {days} days x {seeds} seed(s): {hours:.1f} h')

TypeError: Input 'y' of 'Mul' Op has type float32 that does not match type float64 of argument 'x'.

In [44]:
cmd = [sys.executable, 'run_dnn_dk1.py', '--smoke',
       '--dataset', DATASET, '--datasets-dir', DATASETS, '--out-dir', OUT]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
print('exit code:', result.returncode)

Smoke run: 5 hyperopt evals, 2023-04-11 to 2023-04-13, seed 1.
This validates the pipeline. It is not a model.

Hyperparameter search: 5 evaluations on 2022-04-12..2023-04-10 (before the test period, so no leakage)
Test datasets: 2022-04-12 00:00:00 - 2023-04-10 23:00:00
  268s -> /content/drive/MyDrive/EPF_Masters/experiments/hyperparameters/DNN_hyperparameters_nl2_datDK1_clean_load-windsolar_YT2_SF_CW4_1

Test datasets: 2023-04-11 00:00:00 - 2023-04-13 23:00:00
Imputed missing values (past observations only):
  Price: 8 of 63,840 hours (0.01%) imputed [carry-forward 8, same-hour-earlier-week 0, same-hour-previous-day 0, hour-median 0]
  Exogenous 1: 9 of 63,840 hours (0.01%) imputed [carry-forward 8, same-hour-earlier-week 0, same-hour-previous-day 0, hour-median 0]
      1 hour(s) had no earlier data and were left unfilled
  Exogenous 2: 465 of 63,840 hours (0.73%) imputed [carry-forward 8, same-hour-earlier-week 456, same-hour-previous-day 0, hour-median 0]
      1 hour(s) had no e

In [45]:
import glob, pickle

trials_files = glob.glob(os.path.join(OUT, 'hyperparameters', '*'))
print(trials_files)

with open(trials_files[0], 'rb') as f:
    trials = pickle.load(f)

for t in trials.trials:
    r = t.get('result', {})
    print(t['state'], r.get('status'), r.get('loss'), r.get('MAE Val'))


['/content/drive/MyDrive/EPF_Masters/experiments/hyperparameters/DNN_hyperparameters_nl2_datDK1_clean_load-windsolar_YT2_SF_CW4_1']
2 ok 13.549290228591968 13.549290228591968
2 ok 14.689619254588697 14.689619254588697
2 ok 37.83473771175736 37.83473771175736
2 ok 129.193636965765 129.193636965765
2 ok 19.577582494593457 19.577582494593457


## 5. The real run

The paper's specification: 1500 hyperparameter evaluations, then 728 test days
recalibrated daily, ensembled over 4 seeds.

**Colab disconnects.** Runtimes are reclaimed after idle time and have a session
cap, so a multi-hour run will be interrupted. What survives:

* the **hyperparameter search** checkpoints before every evaluation, so re-running
  with `--skip-hyperopt` picks up the trials file that survived;
* each **seed's forecasts** are written as that seed finishes, so a run that dies
  on seed 3 keeps seeds 1 and 2.

What does not survive is a part-finished seed: there is no per-day checkpointing
inside a seed yet, unlike the LEAR backtest. Chunk long runs with
`--begin-test`/`--end-test` until there is.

In [ ]:
cmd = [sys.executable, 'run_dnn_dk1.py',
       '--dataset', DATASET, '--datasets-dir', DATASETS, '--out-dir', OUT,
       '--max-evals', '1500',
       '--seeds', '1,2,3,4']
print(' '.join(cmd))
# subprocess.run(cmd, check=False)   # uncomment when you mean it

## 6. Scoring

Scored by `lear_dk1.evaluate` — the same code that scores the LEAR runs, not a
parallel copy of it. LEAR ensembles over calibration windows and the DNN over
random seeds, but everything downstream is identical: the ensemble mean, MAE,
rMAE against a weekly seasonal naive, and the multivariate Diebold-Mariano and
Giacomini-White tests of the ensemble against each of its own members.

That shared path is the point. Two numbers computed by the same estimator, over
the same days, against the same observed prices, can be compared; two computed
by separate implementations cannot be trusted to be.

`run_dnn_dk1.py` scores automatically unless `--no-evaluate` is passed. Run this
to score an existing run directory, or to re-score after adding seeds.

In [ ]:
from lear_dk1.evaluate import evaluate_run, format_run
import glob

run_dir = sorted(glob.glob(os.path.join(OUT, f'{DATASET}_dnn_*')))[-1]
print(run_dir, '\n')

results = evaluate_run(run_dir, dataset=DATASET, datasets_dir=DATASETS,
                       zone=DATASET.split('_')[0], kind='seed')

### The predictions

`predictions.csv`, written into the run directory in exactly the layout the LEAR
runs use: one row per hour, the ensemble forecast beside the price that actually
cleared.

```
timestamp_local,zone,forecast,observed
```

Because the two models write the same file, comparing them is a merge on
`timestamp_local` — no reshaping, no re-deriving anything.

In [ ]:
preds = pd.read_csv(os.path.join(run_dir, 'predictions.csv'),
                    parse_dates=['timestamp_local'])
print(preds.shape)
preds.head(24)

### DNN against LEAR

Point `LEAR_RUN` at a LEAR run directory for the same zone and days. The merge
is on the hour, and the DM/GW test below asks whether the difference in loss is
statistically distinguishable rather than just visible.

In [ ]:
LEAR_RUN = os.path.join(OUT, 'DK1_clean_load-windsolar_2023-04-11_2025-04-07')

if os.path.isdir(LEAR_RUN):
    from lear_dk1.evaluate import HOURS, build_ensemble, compare, load_forecasts, real_prices

    lear_preds = pd.read_csv(os.path.join(LEAR_RUN, 'predictions.csv'),
                             parse_dates=['timestamp_local'])
    both = preds.merge(lear_preds, on=['timestamp_local', 'zone'],
                       suffixes=('_dnn', '_lear'))
    # Only hours with an observed price can be scored.
    both = both[both['observed_dnn'].notna()]
    mae_dnn = (both['observed_dnn'] - both['forecast_dnn']).abs().mean()
    mae_lear = (both['observed_dnn'] - both['forecast_lear']).abs().mean()
    print(f'{len(both):,} overlapping hours')
    print(f'  DNN  MAE {mae_dnn:.3f}')
    print(f'  LEAR MAE {mae_lear:.3f}')
else:
    print(f'No LEAR run at {LEAR_RUN} - run the LEAR backtest for this zone first.')